# PatchTST Baseline (Channel-Independent) — electricity + traffic

Trains **canonical PatchTST** (channel-independent: each channel = separate sample, shared weights),
d_model=512, e_layers=3, patch_len=16, stride 8, lookback 512, pred 48.

Channel-independent avoids the O(C²) attention blowup that OOMs on 321/862-channel data.
This matches the original PatchTST paper and is what `benchmark_standard.py` evaluates.

**Runtime**: ~2–5 min total on T4 (all 6 datasets if needed).
**Resumable**: re-run all cells → picks up from last checkpoint.
**Drive**: checkpoints to `MyDrive/nanoforecast-baselines/patchtst/`

## Step 1 — Setup & GPU check

In [ ]:
import torch, sys, os, json, time, shutil
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU.'
torch.set_num_threads(1)
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'git+https://github.com/eulogik/NanoForecast.git@v0.5',
                'safetensors', 'pandas'], check=True)

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/nanoforecast-baselines/patchtst'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## Step 2 — Upload training bundle

Run the cell below, then **upload `nf_bench.tar.gz`** when prompted.
The tarball contains the channel-independent trainer + vendored PatchTST + benchmark harness.

In [ ]:
from google.colab import files
uploaded = files.upload()
if 'nf_bench.tar.gz' in uploaded:
    !tar xzf nf_bench.tar.gz
    print('Extracted. Files:')
    !ls -la benchmarks/ benchmarks/tsl/ train_patchtst_ci.py benchmark_standard.py
else:
    print('ERROR: upload nf_bench.tar.gz first')

## Step 3 — Train electricity + traffic (channel-independent)

In [ ]:
import sys
sys.path.insert(0, '.')
# Point checkpoints to Drive
os.makedirs('/content/checkpoints', exist_ok=True)
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Symlink local checkpoints dir to Drive so trainer writes there
if not os.path.isdir('benchmarks/checkpoints/patchtst'):
    os.makedirs('benchmarks/checkpoints', exist_ok=True)
    if os.path.islink('benchmarks/checkpoints/patchtst'):
        os.unlink('benchmarks/checkpoints/patchtst')
    os.symlink(DRIVE_ROOT, 'benchmarks/checkpoints/patchtst')
    print('Symlinked checkpoints ->', DRIVE_ROOT)

from benchmarks.train_patchtst_ci import train_one

for ds in ['ETTh1', 'ETTh2', 'ETTm1', 'exchange_rate', 'electricity', 'traffic']:
    print(f'\n=== Training {ds} ===')
    r = train_one(ds, 'cuda')
    if r:
        print(json.dumps(r, indent=2))
    torch.cuda.empty_cache()

print('\n=== DONE ===')
print('Checkpoints:', sorted(os.listdir(DRIVE_ROOT)))

## Step 4 — Verify checkpoints on Drive

Download `electricity.pt`, `electricity.json`, `traffic.pt`, `traffic.json` from
`MyDrive/nanoforecast-baselines/patchtst/` to your local
`~/Code/NanoForecast/benchmarks/checkpoints/patchtst/`, then run:
```
python3 benchmark_standard.py --models patchtst --datasets electricity,traffic
```

In [ ]:
for f in sorted(os.listdir(DRIVE_ROOT)):
    sz = os.path.getsize(os.path.join(DRIVE_ROOT, f))
    print(f'  {f:30s} {sz/1024:.1f} KB')

for ds in ['electricity', 'traffic']:
    mp = os.path.join(DRIVE_ROOT, f'{ds}.json')
    if os.path.exists(mp):
        m = json.load(open(mp))
        print(f'{ds}: {m["n_vars"]} vars, {m["epochs"]} epochs, '
              f'best_val_mse={m["best_val_mse"]:.6f}, {m["train_seconds"]}s')
    else:
        print(f'{ds}: NOT FOUND')